# IA LLM Web Services — edição local com Ollama

**PUC-SP · Ciências da Computação · Inteligência Artificial**  
**Autores:** Pedro Henrique Vianna Carvalho (RA00333196) e Helio Masaru Sueta (RA00333246)

Esta edição acompanha os tópicos práticos da aula de APIs de LLM, mas usa um modelo aberto executado pelo Ollama. As chamadas de texto ficam na máquina/ambiente que executa este notebook: não usam créditos da OpenAI nem do Hugging Face e não precisam de chaves de API.

**Importante:** no Colab hospedado, Ollama roda na máquina temporária do próprio Colab, não no seu Windows. A sessão e o modelo baixado são temporários. No VS Code, instale o Ollama para Windows e deixe o serviço iniciado.

## Antes de começar

- **Google Colab:** use um ambiente de execução conectado. GPU pode acelerar, mas não é obrigatória para o modelo pequeno escolhido. Execute a célula de instalação do Ollama abaixo.
- **VS Code/Windows:** instale Ollama em [ollama.com/download](https://ollama.com/download), abra o Ollama para iniciar o serviço e execute `ollama pull qwen2.5:3b` no PowerShell antes de rodar o notebook.
- O modelo `qwen2.5:3b` precisa ser baixado (alguns GB). Com 32 GB de RAM há espaço confortável para começar, embora a velocidade também dependa do processador/GPU.
- Não coloque tokens ou senhas no notebook. As chaves OpenAI/Hugging Face não são necessárias aqui.

Execute as células em ordem. Para desligar/ligar o ambiente Colab, será preciso iniciar o Ollama novamente.

In [ ]:
%pip -q install openai requests pandas seaborn matplotlib pydantic

## 0 · Iniciar Ollama e baixar o modelo

No Colab Linux, esta célula instala Ollama e inicia o servidor local. Em Windows, ela espera que Ollama já esteja instalado e aberto. O download acontece uma vez por ambiente.

In [ ]:
import os
import platform
import shutil
import subprocess
import time

import requests

OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen2.5:3b"
ollama_process = None

def ollama_is_ready():
    try:
        return requests.get(OLLAMA_URL, timeout=2).ok
    except requests.RequestException:
        return False

if not ollama_is_ready():
    if "COLAB_RELEASE_TAG" in os.environ and platform.system() == "Linux":
        install = subprocess.run(
            ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
            text=True,
            capture_output=True,
        )
        if install.returncode != 0:
            raise RuntimeError(f"Instalação do Ollama falhou: {install.stderr[-2000:]}")
        ollama_process = subprocess.Popen(
            ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
        )
    elif shutil.which("ollama"):
        ollama_process = subprocess.Popen(
            ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT
        )
    else:
        raise RuntimeError(
            "Ollama não está acessível. No Windows, instale e abra Ollama; "
            "no Colab, conecte um ambiente Linux e execute novamente."
        )

for _ in range(60):
    if ollama_is_ready():
        break
    if ollama_process is not None and ollama_process.poll() is not None:
        raise RuntimeError("O processo Ollama encerrou antes de iniciar o servidor.")
    time.sleep(1)
else:
    raise RuntimeError("Ollama não respondeu em http://127.0.0.1:11434 após 60 segundos.")

print("Servidor Ollama pronto:", OLLAMA_URL)
pull = subprocess.run(["ollama", "pull", MODEL], text=True)
if pull.returncode != 0:
    raise RuntimeError(f"Não foi possível baixar {MODEL}. Confira a conexão e o espaço em disco.")
print("Modelo disponível:", MODEL)

## 1 · Anatomia de uma chamada HTTP/JSON

O Ollama oferece uma API local compatível com o formato de Chat Completions da OpenAI. O endereço é local (`127.0.0.1`); a chamada não vai para a OpenAI.

In [ ]:
import json

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": "Em uma frase, o que é uma API web?"}],
    "stream": False,
}
response = requests.post(f"{OLLAMA_URL}/v1/chat/completions", json=payload, timeout=180)
response.raise_for_status()
data = response.json()
print("HTTP status:", response.status_code)
print(json.dumps(data, ensure_ascii=False, indent=2)[:2500])

## 2 · Cliente compatível e helper de chat

A biblioteca `openai` simplifica a comunicação, mas o `base_url` aponta exclusivamente para Ollama. Tokens e tempos são medidos localmente; o provedor não fornece preço nesta configuração.

In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(base_url=f"{OLLAMA_URL}/v1", api_key="ollama", timeout=180.0)

def chat(messages, **params):
    """Chama o modelo Ollama configurado e retorna texto e resposta bruta."""
    result = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        **params,
    )
    text = result.choices[0].message.content or ""
    return text, result

answer, raw = chat([
    {"role": "system", "content": "Você é um monitor de Ciência da Computação. Responda em português e seja conciso."},
    {"role": "user", "content": "Por que APIs de LLM costumam medir uso em tokens?"},
])
display(Markdown(answer))
if raw.usage:
    print(f"\nTokens de entrada: {raw.usage.prompt_tokens} | saída: {raw.usage.completion_tokens}")
else:
    print("Contagem de tokens não foi informada por esta versão do servidor/modelo.")

## 3 · Papéis, mensagens e memória

A API não guarda contexto automaticamente: para continuar uma conversa, o programa envia o histórico novamente.

In [ ]:
first, _ = chat([{"role": "user", "content": "Meu nome é Ada e estudo na PUC-SP. Dê oi."}])
second, _ = chat([{"role": "user", "content": "Qual é meu nome e onde estudo?"}])
with_history, _ = chat([
    {"role": "user", "content": "Meu nome é Ada e estudo na PUC-SP. Dê oi."},
    {"role": "assistant", "content": first},
    {"role": "user", "content": "Qual é meu nome e onde estudo?"},
])
print("Sem histórico (a resposta pode variar):", second)
print("Com histórico:", with_history)

## 4 · Streaming

Com `stream=True`, o texto chega em pedaços. O modelo ainda roda localmente; a velocidade depende do hardware.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explique em três frases o que são tokens em um LLM."}],
    stream=True,
    temperature=0.4,
)
for chunk in stream:
    piece = chunk.choices[0].delta.content
    if piece:
        print(piece, end="", flush=True)
print()

## 5 · Temperatura e amostragem

A temperatura controla a variação das respostas. Vamos repetir a mesma pergunta e comparar as saídas.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

question = "Sugira um nome criativo para um aplicativo que ajuda estudantes a organizar tarefas."
samples = []
for temperature in (0.0, 0.7, 1.2):
    text, _ = chat([{"role": "user", "content": question}], temperature=temperature, max_tokens=100)
    samples.append({"temperatura": temperature, "resposta": text.strip()})

temperature_df = pd.DataFrame(samples)
display(temperature_df)
sns.barplot(data=temperature_df, x="temperatura", y=temperature_df["resposta"].str.len())
plt.title("Tamanho das respostas por temperatura")
plt.ylabel("Caracteres (apenas referência)")
plt.show()

## 6 · Saída estruturada e pandas

Pedimos JSON e validamos com Pydantic antes de usar os dados. O modelo pode variar; uma falha de formato deve ser corrigida ou reportada, não tratada como dado válido.

In [ ]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str
    category: str
    monthly_cost_brl: float = Field(ge=0)

class ProductList(BaseModel):
    products: list[Product]

schema_prompt = (
    "Retorne somente um objeto JSON válido, sem Markdown, no formato "
    '{"products":[{"name":"...","category":"...","monthly_cost_brl":0.0}]}. '
    "Crie três serviços fictícios de streaming com custos mensais plausíveis em reais."
)
json_text, _ = chat([{"role": "user", "content": schema_prompt}],
                    response_format={"type": "json_object"}, temperature=0.2)
products = ProductList.model_validate_json(json_text)
products_df = pd.DataFrame([item.model_dump() for item in products.products])
display(products_df)
sns.barplot(data=products_df, x="name", y="monthly_cost_brl")
plt.title("Custos fictícios informados pelo modelo")
plt.ylabel("R$/mês")
plt.xticks(rotation=20)
plt.show()

## 7 · Tool calling: consulta SQL somente leitura

O modelo sugere uma chamada de ferramenta; o Python valida e executa a consulta. O banco permanece sob controle do aplicativo. A conexão abaixo está em modo somente leitura e aceita apenas uma instrução `SELECT`.

In [ ]:
import sqlite3
import json

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE grades (student TEXT, course TEXT, grade REAL);
INSERT INTO grades VALUES
('Ana','Algorithms',8.5), ('Ana','Databases',9.0), ('Ana','AI',9.5),
('Bruno','Algorithms',6.0), ('Bruno','Databases',7.5), ('Bruno','AI',5.5),
('Carla','Algorithms',9.5), ('Carla','Databases',8.0), ('Carla','AI',10.0);
""")
db.execute("PRAGMA query_only = ON")

def run_sql(query: str) -> str:
    normalized = query.strip().lower()
    if not normalized.startswith("select") or ";" in normalized.rstrip(";"):
        return "ERROR: somente uma consulta SELECT é permitida."
    try:
        cursor = db.execute(query)
        return json.dumps(cursor.fetchall(), ensure_ascii=False)
    except sqlite3.Error as exc:
        return f"ERROR: consulta recusada: {exc}"

sql_tool = [{
    "type": "function",
    "function": {
        "name": "run_sql",
        "description": "Executa uma consulta SELECT somente leitura no banco de notas da universidade.",
        "parameters": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
}]

messages = [
    {"role": "system", "content": "Use run_sql para consultar notas. Nunca invente resultados."},
    {"role": "user", "content": "Qual é a média de cada estudante?"},
]
trace = [("App", "LLM", "pergunta + ferramenta")]
result = client.chat.completions.create(model=MODEL, messages=messages, tools=sql_tool, tool_choice="auto")
assistant_message = result.choices[0].message
messages.append(assistant_message.model_dump(exclude_none=True))

if not assistant_message.tool_calls:
    print(assistant_message.content or "O modelo não solicitou ferramenta; tente novamente.")
else:
    for tool_call in assistant_message.tool_calls:
        if tool_call.function.name != "run_sql":
            raise ValueError(f"Ferramenta não permitida: {tool_call.function.name}")
        args = json.loads(tool_call.function.arguments)
        sql_result = run_sql(args["query"])
        trace.append(("LLM", "App", args["query"]))
        trace.append(("App", "SQLite", "SELECT validado"))
        trace.append(("SQLite", "App", sql_result))
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": sql_result})
    final = client.chat.completions.create(model=MODEL, messages=messages)
    trace.append(("App", "LLM", "resultado da consulta"))
    print(final.choices[0].message.content)

for src, dst, label in trace:
    print(f"{src} -> {dst}: {label[:90]}")

## 8 · Multimodal: imagem como entrada (opcional)

O modelo de texto padrão não analisa imagens. Para testar visão, baixe `qwen2.5vl:7b` e configure um arquivo de imagem acessível ao kernel. Em Colab, carregue o arquivo para `/content`; no Windows, informe o caminho local. Este modelo maior pode exigir bastante memória e espaço. Não há geração de imagem nesta edição: ela depende de um serviço/modelo de geração separado.

In [ ]:
import base64
from pathlib import Path

VISION_MODEL = "qwen2.5vl:7b"
IMAGE_PATH = ""  # Ex.: /content/grafico.png no Colab ou C:\\Users\\pedro\\Pictures\\grafico.png no Windows

if not IMAGE_PATH:
    print("Análise de imagem pulada. Defina IMAGE_PATH para executar esta demonstração.")
else:
    image_file = Path(IMAGE_PATH)
    if not image_file.is_file():
        raise FileNotFoundError(f"Imagem não encontrada: {image_file}")
    pull = subprocess.run(["ollama", "pull", VISION_MODEL], text=True)
    if pull.returncode != 0:
        raise RuntimeError(f"Não foi possível baixar o modelo de visão {VISION_MODEL}.")
    encoded = base64.b64encode(image_file.read_bytes()).decode("ascii")
    mime = "image/png" if image_file.suffix.lower() == ".png" else "image/jpeg"
    vision_result = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "Descreva esta imagem em três observações concisas."},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{encoded}"}},
        ]}],
    )
    print(vision_result.choices[0].message.content)

## 9 · Benchmark local: latência e tokens

Comparamos tarefas no mesmo modelo e computador. O valor monetário não é calculado porque a inferência é local; energia, hardware e tempo continuam tendo custo.

In [ ]:
import time

prompts = {
    "fato": "O que é busca binária? Responda em uma frase.",
    "código": "Escreva uma função Python que verifica se uma string é palíndromo. Apenas código.",
    "tradução": "Traduza para português: The model returned a response in two seconds.",
}
rows = []
for task, prompt in prompts.items():
    start = time.perf_counter()
    text, result = chat([{"role": "user", "content": prompt}], temperature=0.2, max_tokens=160)
    seconds = time.perf_counter() - start
    usage = result.usage
    out_tokens = usage.completion_tokens if usage else None
    rows.append({"tarefa": task, "segundos": seconds, "tokens_saida": out_tokens,
                 "tokens_por_segundo": out_tokens / seconds if out_tokens is not None and seconds else None,
                 "resposta": " ".join(text.split())[:100]})

benchmark_df = pd.DataFrame(rows)
display(benchmark_df.round(2))
sns.barplot(data=benchmark_df, x="tarefa", y="segundos")
plt.title(f"Latência do modelo local {MODEL}")
plt.ylabel("Segundos (menor é melhor)")
plt.show()

## Encerramento e checklist

- Uma API compatível pode ser servida localmente; `base_url` determina para onde o cliente envia as chamadas.
- O histórico precisa ser incluído para manter contexto; streaming mostra trechos conforme são gerados.
- Saídas estruturadas devem ser validadas; chamadas de ferramentas devem ser verificadas pelo aplicativo.
- A análise de imagem usa outro modelo local. Geração de imagem não está incluída.
- No Colab, baixar o Ollama/modelo usa a sessão e a conexão de rede; depois, as inferências são feitas pelo servidor local dessa sessão.

**Atividade:** compare uma resposta com temperaturas `0.0` e `1.0`, descreva a diferença, e desenhe o limite de confiança no exemplo SQL: o modelo propõe; o programa valida; somente então o banco responde.